In [11]:
import geopandas as gpd

In [12]:
gdf = gpd.read_file("../../data/matchup_indices_s2cloudless.geojson")
#gdf = gpd.read_file("../../data/matchup_indices_omnicloud.geojson")
gdf.head(2)

,fecha,fuente,chla,grupo_nombre,estado_trofico,delta,pass_id,sample_id,ndci_median,ndci_mean,...,B05_median,B05_mean,B06_median,B06_mean,B07_median,B07_mean,paso_cloudless,n_pixeles_usados,ventana_recortada,geometry
0,2017-01-02,GEMS,6.8,LDS,O,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,1,-0.046154,-0.045572,...,0.0650,0.065052,0.0417,0.041656,0.0410,0.04102,True,25.0,False,POINT (678380.002 6147956.963)
1,2017-01-02,GEMS,3.2,LDS,O,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,2,-0.048656,-0.049677,...,0.0646,0.064852,0.0425,0.042280,0.0423,0.04216,True,25.0,False,POINT (679045.962 6144090.994)


In [13]:
#gdf[(gdf["n_pixeles_usados"] == 0)]["ndci"].isna().sum()

In [14]:
import xarray as xr
import numpy as np
from pathlib import Path

# "no data" rows: the crop is 100% nodata (raw DN=0 -> reflectance floor -0.1),
# not detectable from the geojson columns alone -- ndci/indice_combinado can look
# like plausible numbers even when every pixel is nodata, so check the .nc files directly
BANDS_DIR = Path("../../data/bands/B07-B06-B05-B04-B03-B02-B01-B08-B8A-B09-B10-B11-B12")
full_nodata_indices = set()
for f in BANDS_DIR.glob("shard-*/*.nc"):
    ds = xr.open_dataset(f)
    b04 = ds["B04"].isel(time=0).values
    if np.isclose(b04, -0.1, atol=1e-6).all():
        full_nodata_indices.add(int(f.stem))
    ds.close()

nodata_rows = gdf.loc[gdf.index.isin(full_nodata_indices)]
nodata_rows

,fecha,fuente,chla,grupo_nombre,estado_trofico,delta,pass_id,sample_id,ndci_median,ndci_mean,...,B05_median,B05_mean,B06_median,B06_mean,B07_median,B07_mean,paso_cloudless,n_pixeles_usados,ventana_recortada,geometry
102,2017-03-16 00:00:00,GEMS,1.6,LDS,U,NaN,NaN,164,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (676618.974 6142887.026)
103,2017-03-23 00:00:00,GEMS,5.7,LDS,O,0 days 10:17:58.974000,S2A_MSIL1C_20170322T134201_N0500_R124_T21HXB_2...,165,-0.047527,-0.042544,...,0.0709,0.078050,0.06675,0.076694,0.0701,0.081606,True,18.0,False,POINT (671890.038 6142737.153)
105,2017-04-03 00:00:00,OAN,5.9,RDP-MONTES,O,1 days 13:51:11.026000,S2A_MSIL1C_20170404T135111_N0500_R024_T21HUC_2...,179,-0.033379,-0.023759,...,0.1442,0.144522,0.10360,0.111978,0.1133,0.123250,True,18.0,False,POINT (401164.986 6214585.025)
110,2017-04-03 00:00:00,OAN,3.0,RDP-MONTES,O,1 days 13:51:11.026000,S2A_MSIL1C_20170404T135111_N0500_R024_T21HUC_2...,184,-0.050566,-0.049487,...,0.1385,0.138376,0.07800,0.078240,0.0826,0.082252,True,25.0,False,POINT (392250.023 6218593.96)
111,2017-04-24 00:00:00,OAN,3.0,RDP-MONTES,O,0 days 13:51:11.026000,S2A_MSIL1C_20170424T135111_N0500_R024_T21HUC_2...,192,-0.047840,-0.047955,...,0.1014,0.101492,0.05870,0.058724,0.0583,0.058152,True,25.0,False,POINT (402233.023 6210401.038)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,2024-12-03 11:23:00,OAN,15.0,RN-UPM1,M,0 days 02:24:01.024000,S2A_MSIL1C_20241203T134701_N0511_R024_T21HUD_2...,2744,-0.024103,-0.021493,...,0.0842,0.084572,0.06140,0.064512,0.0619,0.065928,True,25.0,False,POINT (381960.791 6335871.055)
1601,2024-12-04 00:00:00,OAN,40.3,LDS,E,1 days 13:38:39.024000,S2B_MSIL1C_20241205T133839_N0511_R124_T21HXB_2...,2745,-0.015660,-0.018761,...,0.0844,0.083504,0.06830,0.066112,0.0682,0.066344,True,25.0,False,POINT (679045.962 6144090.994)
1602,2024-12-04 07:52:00,OAN,3.0,RN-UPM2,O,1 days 05:46:39.024000,S2B_MSIL1C_20241205T133839_N0511_R124_T21HWD_2...,2746,-0.036916,-0.037660,...,0.0553,0.055292,0.03920,0.039508,0.0394,0.039296,True,25.0,False,POINT (518527.361 6362411.37)
1603,2024-12-04 08:30:00,OAN,11.0,RN-UPM1,M,0 days 18:29:18.975000,S2C_MSIL1C_20241203T140041_N0511_R024_T21HUD_2...,2747,-0.035848,-0.036126,...,0.0895,0.089112,0.06650,0.066448,0.0654,0.065372,True,25.0,False,POINT (370210.474 6329546.047)


In [15]:
gdf[(gdf["paso_cloudless"] == 'False')]

,fecha,fuente,chla,grupo_nombre,estado_trofico,delta,pass_id,sample_id,ndci_median,ndci_mean,...,B05_median,B05_mean,B06_median,B06_mean,B07_median,B07_mean,paso_cloudless,n_pixeles_usados,ventana_recortada,geometry
747,2022-01-05 06:05:00,OAN,1.92,LDS,U,0 days 07:37:11.024000,S2A_MSIL1C_20220105T134211_N0511_R124_T21HXB_2...,1517,-0.041778,-0.040513,...,0.09050,0.088996,0.07320,0.074240,0.0735,0.074964,False,25.0,False,POINT (679045.962 6144090.994)
748,2022-01-06 09:06:00,GEMS,4.70,LDS,O,0 days 19:23:48.976000,S2A_MSIL1C_20220105T134211_N0511_R124_T21HXB_2...,1518,0.021395,0.052941,...,0.05560,0.058542,0.06700,0.077632,0.0721,0.085863,False,19.0,False,POINT (677080.109 6150100.27)
749,2022-01-06 09:25:00,GEMS,4.10,LDS,O,0 days 19:42:48.976000,S2A_MSIL1C_20220105T134211_N0511_R124_T21HXB_2...,1519,0.023476,0.024710,...,0.41605,0.436937,0.43870,0.459338,0.4530,0.473463,False,8.0,False,POINT (678718.784 6151822.323)
750,2022-01-06 10:00:00,GEMS,4.50,LDS,O,0 days 20:17:48.976000,S2A_MSIL1C_20220105T134211_N0511_R124_T21HXB_2...,1520,-0.030602,-0.031325,...,0.09670,0.096168,0.07550,0.074944,0.0746,0.074252,False,25.0,False,POINT (678380.002 6147956.963)
751,2022-01-06 10:25:00,GEMS,3.50,LDS,O,0 days 20:42:48.976000,S2A_MSIL1C_20220105T134211_N0511_R124_T21HXB_2...,1521,-0.016462,-0.016435,...,0.12400,0.151496,0.11010,0.140820,0.1083,0.144312,False,25.0,False,POINT (676618.974 6142887.026)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,2025-07-24 00:00:00,OAN,0.10,RDP-MONTES,U,0 days 10:08:28.976000,S2A_MSIL1C_20250723T135131_N0511_R024_T21HUC_2...,2947,-0.046193,-0.046774,...,0.15450,0.153132,0.11380,0.111804,0.1124,0.110408,False,25.0,False,POINT (402625.965 6210316.961)
1764,2025-07-24 00:00:00,OAN,0.10,RDP-MONTES,U,0 days 10:08:28.976000,S2A_MSIL1C_20250723T135131_N0511_R024_T21HUC_2...,2948,-0.063733,-0.063431,...,0.06260,0.062548,0.04390,0.043788,0.0408,0.040808,False,25.0,False,POINT (387237.037 6218893.952)
1765,2025-07-24 00:00:00,OAN,1.50,RDP-MONTES,U,0 days 10:08:28.976000,S2A_MSIL1C_20250723T135131_N0511_R024_T21HUC_2...,2949,-0.001079,-0.004944,...,0.12590,0.129535,0.12375,0.122675,0.1301,0.127230,False,20.0,False,POINT (401164.986 6214585.025)
1766,2025-07-24 00:00:00,OAN,1.50,RDP-MONTES,U,0 days 10:08:28.976000,S2A_MSIL1C_20250723T135131_N0511_R024_T21HUC_2...,2950,-0.044723,-0.043965,...,0.10770,0.108348,0.06550,0.065648,0.0629,0.063428,False,25.0,False,POINT (392250.022 6218594.071)


In [16]:
# clean, trustworthy dataset: drop the no-data (tile-boundary bug) rows and the
# ones that never went through cloud filtering (missing B10)
clean_gdf = gdf.loc[
    ~gdf.index.isin(full_nodata_indices) &
    #(gdf["paso_cloudless"] == "True") &
    (gdf["n_pixeles_usados"] != 0.0)
].copy()

print(f"clean_gdf: {len(clean_gdf)} rows (from {len(gdf)} total)")
print(f"still NaN in clean_gdf (fully cloudy, 0 clear pixels -- not removed, just flagging): {clean_gdf['ndci_mean'].isna().sum()}")
clean_gdf

clean_gdf: 1625 rows (from 1819 total)
still NaN in clean_gdf (fully cloudy, 0 clear pixels -- not removed, just flagging): 50


,fecha,fuente,chla,grupo_nombre,estado_trofico,delta,pass_id,sample_id,ndci_median,ndci_mean,...,B05_median,B05_mean,B06_median,B06_mean,B07_median,B07_mean,paso_cloudless,n_pixeles_usados,ventana_recortada,geometry
0,2017-01-02 00:00:00,GEMS,6.8,LDS,O,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,1,-0.046154,-0.045572,...,0.06500,0.065052,0.04170,0.041656,0.04100,0.041020,True,25.0,False,POINT (678380.002 6147956.963)
1,2017-01-02 00:00:00,GEMS,3.2,LDS,O,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,2,-0.048656,-0.049677,...,0.06460,0.064852,0.04250,0.042280,0.04230,0.042160,True,25.0,False,POINT (679045.962 6144090.994)
2,2017-01-02 00:00:00,GEMS,5.9,LDS,O,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,5,-0.049635,-0.049793,...,0.06490,0.064848,0.04090,0.040888,0.04010,0.040212,True,25.0,False,POINT (676618.974 6142887.026)
3,2017-01-02 00:00:00,GEMS,11.4,LDS,M,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,6,-0.034286,-0.031558,...,0.03490,0.034848,0.02870,0.029100,0.02800,0.028248,True,25.0,False,POINT (669756.975 6142832.05)
4,2017-01-02 00:00:00,GEMS,6.4,LDS,O,3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...,9,-0.045675,-0.045992,...,0.05130,0.051848,0.03650,0.036740,0.03570,0.036156,True,25.0,False,POINT (671890.038 6142737.153)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1814,2026-03-04 15:53:00,OAN,3.0,RN-UPM2,O,0 days 21:45:31.025000,S2C_MSIL1C_20260305T133831_N0512_R124_T21HWD_2...,3060,-0.083676,-0.063844,...,0.03340,0.038160,0.03080,0.037484,0.02970,0.037576,True,25.0,False,POINT (541076.43 6365906.268)
1815,2026-05-06 12:51:00,OAN,3.0,RN-UPM2,O,0 days 00:51:21.024000,S2A_MSIL1C_20260506T134221_N0512_R124_T21HWD_2...,3082,-0.079365,-0.073953,...,0.03480,0.035960,0.03010,0.032604,0.02960,0.033104,True,25.0,False,POINT (528098.467 6367260.472)
1816,2026-05-06 15:14:00,OAN,1.5,RN-UPM2,U,0 days 01:31:38.976000,S2A_MSIL1C_20260506T134221_N0512_R124_T21HWD_2...,3084,-0.096467,-0.080931,...,0.05655,0.057270,0.04150,0.048660,0.04230,0.050450,True,10.0,False,POINT (540411.19 6365961.336)
1817,2026-05-06 15:28:00,OAN,3.0,RN-UPM2,O,0 days 01:45:38.976000,S2A_MSIL1C_20260506T134221_N0512_R124_T21HWD_2...,3085,-0.097728,-0.090880,...,0.05225,0.053368,0.04045,0.042936,0.04135,0.043614,True,22.0,False,POINT (540435.203 6366112.119)


In [17]:
df = clean_gdf.drop(columns=["pass_id", "n_pixeles_usados", "ventana_recortada", "delta", "fuente", "paso_cloudless", "sample_id"])
df

,fecha,chla,grupo_nombre,estado_trofico,ndci_median,ndci_mean,ndci_iqr,ndci_std,indice_tres_bandas_median,indice_tres_bandas_mean,...,B03_mean,B04_median,B04_mean,B05_median,B05_mean,B06_median,B06_mean,B07_median,B07_mean,geometry
0,2017-01-02 00:00:00,6.8,LDS,O,-0.046154,-0.045572,0.002933,0.002157,0.000026,0.000025,...,0.081760,0.07130,0.071264,0.06500,0.065052,0.04170,0.041656,0.04100,0.041020,POINT (678380.002 6147956.963)
1,2017-01-02 00:00:00,3.2,LDS,O,-0.048656,-0.049677,0.002722,0.003910,0.000029,0.000029,...,0.081296,0.07160,0.071632,0.06460,0.064852,0.04250,0.042280,0.04230,0.042160,POINT (679045.962 6144090.994)
2,2017-01-02 00:00:00,5.9,LDS,O,-0.049635,-0.049793,0.003003,0.001746,0.000027,0.000027,...,0.081816,0.07160,0.071644,0.06490,0.064848,0.04090,0.040888,0.04010,0.040212,POINT (676618.974 6142887.026)
3,2017-01-02 00:00:00,11.4,LDS,M,-0.034286,-0.031558,0.007362,0.011089,0.000007,0.000006,...,0.059328,0.03700,0.037108,0.03490,0.034848,0.02870,0.029100,0.02800,0.028248,POINT (669756.975 6142832.05)
4,2017-01-02 00:00:00,6.4,LDS,O,-0.045675,-0.045992,0.003620,0.004320,0.000017,0.000018,...,0.075836,0.05570,0.056852,0.05130,0.051848,0.03650,0.036740,0.03570,0.036156,POINT (671890.038 6142737.153)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1814,2026-03-04 15:53:00,3.0,RN-UPM2,O,-0.083676,-0.063844,0.016913,0.051428,0.000018,0.000013,...,0.064140,0.03950,0.043148,0.03340,0.038160,0.03080,0.037484,0.02970,0.037576,POINT (541076.43 6365906.268)
1815,2026-05-06 12:51:00,3.0,RN-UPM2,O,-0.079365,-0.073953,0.014042,0.024110,0.000018,0.000017,...,0.062216,0.04120,0.041628,0.03480,0.035960,0.03010,0.032604,0.02960,0.033104,POINT (528098.467 6367260.472)
1816,2026-05-06 15:14:00,1.5,RN-UPM2,U,-0.096467,-0.080931,0.029251,0.030697,0.000046,0.000043,...,0.089700,0.06510,0.067050,0.05655,0.057270,0.04150,0.048660,0.04230,0.050450,POINT (540411.19 6365961.336)
1817,2026-05-06 15:28:00,3.0,RN-UPM2,O,-0.097728,-0.090880,0.015290,0.020470,0.000045,0.000044,...,0.086214,0.06455,0.063900,0.05225,0.053368,0.04045,0.042936,0.04135,0.043614,POINT (540435.203 6366112.119)


In [18]:
df.groupby("estado_trofico").count()

,fecha,chla,grupo_nombre,ndci_median,ndci_mean,ndci_iqr,ndci_std,indice_tres_bandas_median,indice_tres_bandas_mean,indice_tres_bandas_iqr,...,B03_mean,B04_median,B04_mean,B05_median,B05_mean,B06_median,B06_mean,B07_median,B07_mean,geometry
estado_trofico,,,,,,,,,,,,,,,,,,,,,
E,37,37,37,37,37,37,37,37,37,37,...,37,37,37,37,37,37,37,37,37,37
M,265,265,265,260,260,260,260,260,260,260,...,260,260,260,260,260,260,260,260,260,265
O,806,806,806,787,787,787,787,787,787,787,...,787,787,787,787,787,787,787,787,787,806
U,517,517,517,491,491,491,491,491,491,491,...,491,491,491,491,491,491,491,491,491,517


In [19]:
gdf["n_pixeles_usados"].min(),gdf["n_pixeles_usados"].max()

(np.float64(5.0), np.float64(25.0))

In [20]:
df.to_file("../../data/chla_indices_v1.geojson", driver="GeoJson", mode="w")